<a href="https://colab.research.google.com/github/runessaa/-Streltsov-Projects/blob/main/%D0%9F%D1%80%D0%B0%D0%BA%D1%82%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%B0%D1%8F_%D1%80%D0%B0%D0%B1%D0%BE%D1%82%D0%B0_%E2%84%9610_%D0%A0%D0%B0%D0%B7%D0%B2%D0%B5%D0%B4%D0%BE%D1%87%D0%BD%D1%8B%D0%B9_%D0%B0%D0%BD%D0%B0%D0%BB%D0%B8%D0%B7%2C_%D0%BF%D1%80%D0%B5%D0%B4%D0%BE%D0%B1%D1%80%D0%B0%D0%B1%D0%BE%D1%82%D0%BA%D0%B0_%D0%B4%D0%B0%D0%BD%D0%BD%D1%8B%D1%85_%D0%B8_%D0%BE%D0%B1%D1%83%D1%87%D0%B5%D0%BD%D0%B8%D0%B5_%D0%BC%D0%BE%D0%B4%D0%B5%D0%BB%D0%B5%D0%B9_%D0%BA%D0%BB%D0%B0%D1%81%D1%81%D0%B8%D1%84%D0%B8%D0%BA%D0%B0%D1%86%D0%B8%D0%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Практическая работа №10. Разведочный анализ, предобработка данных и обучение моделей классификации**

*(на примере датасета Bank Marketing)*

---

## ****Введение****

**Цель работы:** Обучить модель машинного обучения для предсказания, откроет ли клиент банка срочный вклад по результатам маркетинговой кампании (целевая переменная `y`: "yes" — откроет, "no" — не откроет).

**Датасет:** [Bank Marketing Dataset](https://archive.ics.uci.edu/dataset/222/bank+marketing) — данные о маркетинговых кампаниях португальского банка.

**Что вы научитесь делать:**
- Загружать и исследовать данные
- Выявлять проблемы через разведочный анализ (EDA)
- Анализировать и обрабатывать пропущенные значения
- Формулировать и **экспериментально проверять** гипотезы
- Создавать новые признаки (Feature Engineering)
- **Сравнивать все изученные методы классификации**
- **Выбирать лучшие модели и оптимизировать их гиперпараметры с помощью GridSearchCV**
- **Анализировать влияние комбинаций признаков на качество модели**

---

## ****Часть 1. Настройка окружения****

> 📚 **Подсказка:** Аналогичная настройка рассматривалась в [**Занятии 1, Часть 1** и **Занятии 2, Часть 1**](https://colab.research.google.com/drive/1Cnp7hPUUu5dorlhPPHUeztnnoq5Z2j4W?usp=sharing#scrollTo=5f49d9aa).

In [ ]:
# Импорт библиотек
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                            f1_score, roc_auc_score, roc_curve, confusion_matrix,
                            classification_report)

# Модели классификации
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                              AdaBoostClassifier)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

# Отбор признаков
from sklearn.feature_selection import RFE, SelectKBest, f_classif
from itertools import combinations

# Настройки отображения
sns.set_style("whitegrid")
pd.set_option('display.max_columns', 20)

---

## ****Часть 2. Загрузка и первичный осмотр данных****

> 📚 **Подсказка:** Методы первичного осмотра (`head()`, `info()`, `describe()`, `shape`) подробно разобраны в [**Занятии 1, Часть 3.2**](https://colab.research.google.com/drive/1Cnp7hPUUu5dorlhPPHUeztnnoq5Z2j4W?usp=sharing#scrollTo=9ef7543b) и [**Занятии 2, Часть 2.1**](https://colab.research.google.com/drive/1aT36P0Jc-dIjYOx6Hh8yecFSgt-z3Fan?usp=sharing#scrollTo=43b439e3).

### ****Задание 2.1. Загрузите датасет****

In [ ]:
# Загрузка датасета Bank Marketing
url = "https://raw.githubusercontent.com/rishabhathiya/Bank-Marketing/refs/heads/main/bank.csv"
bank = pd.read_csv(url, sep=';')

# ВАЖНО: В этом датасете пропуски записаны как "unknown"
# Заменяем их на NaN, чтобы isna() корректно их определял
bank = bank.replace('unknown', np.nan)

bank.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,30,unemployed,married,primary,no,1787,no,no,cellular,19,oct,79,1,-1,0,NaN,no
1,33,services,married,secondary,no,4789,yes,yes,cellular,11,may,220,1,339,4,failure,no
2,35,management,single,tertiary,no,1350,yes,no,cellular,16,apr,185,1,330,1,failure,no
3,30,management,married,tertiary,no,1476,yes,yes,NaN,3,jun,199,4,-1,0,NaN,no
4,59,blue-collar,married,secondary,no,0,yes,no,NaN,5,may,226,1,-1,0,NaN,no


### ****Задание 2.2. Изучите структуру данных****

**1. Выведите размерность датасета:**

In [ ]:
bank.shape

**2. Отобразите первые 10 строк датасета**

In [ ]:
bank.head(10)

**3. Выведите информацию о типах данных и пропусках**

In [ ]:
bank.info()

**4. Выведите статистику числовых признаков**

In [ ]:
bank.describe()

**5. Выведите названия всех столбцов в виде списка строк**

In [ ]:
list(bank.columns)

### ****Задание 2.3. Ответьте на вопросы****

---

> 📝 **Как отвечать на текстовые вопросы?** Дважды кликните на ячейку → впишите ответ → нажмите `Shift+Enter`

---

**Вопрос 1:** Сколько записей (клиентов) в датасете?

**Ответ:** 4521 запись.

**Вопрос 2:** Сколько признаков (столбцов)?

**Ответ:** 17 столбцов в исходном датасете.

**Вопрос 3:** Какая переменная является целевой (что предсказываем)?

**Ответ:** `y` — откроет ли клиент срочный вклад (`yes` / `no`).

**Вопрос 4:** Какие типы данных преобладают — числовые или категориальные?

**Ответ:** Преобладают категориальные признаки: их больше, чем числовых.

---

### ****Описание столбцов датасета Bank Marketing****

| Столбец | Описание | Тип |
|---------|----------|-----|
| `age` | Возраст клиента | Числовой |
| `job` | Тип занятости | Категориальный |
| `marital` | Семейное положение | Категориальный |
| `education` | Уровень образования | Категориальный |
| `default` | Есть ли дефолт по кредиту | Категориальный |
| `balance` | Баланс на счёте (евро) | Числовой |
| `housing` | Есть ли ипотека | Категориальный |
| `loan` | Есть ли личный кредит | Категориальный |
| `contact` | Тип связи | Категориальный |
| `day` | День последнего контакта | Числовой |
| `month` | Месяц последнего контакта | Категориальный |
| `duration` | Длительность последнего звонка (сек) | Числовой |
| `campaign` | Кол-во контактов в этой кампании | Числовой |
| `pdays` | Дней с последнего контакта (-1 = не было) | Числовой |
| `previous` | Кол-во контактов до этой кампании | Числовой |
| `poutcome` | Результат прошлой кампании | Категориальный |
| `y` | **Целевая переменная:** открыл ли вклад | Категориальный |

---

## ****Часть 3. Разведочный анализ данных (EDA)****

> 📚 **Подсказка:** Цель EDA — выявить проблемы в данных и найти закономерности (паттерны). Методология описана в [**Занятии 2, Часть 2**](https://colab.research.google.com/drive/1aT36P0Jc-dIjYOx6Hh8yecFSgt-z3Fan?usp=sharing#scrollTo=29f19e7a).

### ****3.1. Анализ целевой переменной (баланс классов)****

> 📚 **Подсказка:** Анализ баланса классов рассмотрен в [**Занятии 2, Часть 2.3**](https://colab.research.google.com/drive/1aT36P0Jc-dIjYOx6Hh8yecFSgt-z3Fan?usp=sharing#scrollTo=15c684d7).

In [ ]:
# Создаём числовую версию целевой переменной в столбце target
bank['target'] = (bank['y'] == 'yes').astype(int)

bank.head()

**Подсчитайте распределение целевой переменной 'y'**

In [ ]:
bank['y'].value_counts()

**Вопрос:** сколько клиентов открыли депозит (yes)? сколько не открыли (no)?

**Ответ:** `yes` — 521 клиент, `no` — 4000 клиентов.

---

**Визуализируйте распределения классов графически (постройте столбчатую (barplot) или круговую (pie chart) диаграмму)**

In [ ]:
sns.countplot(data=bank, x='y')
plt.title('Распределение целевой переменной')
plt.xlabel('Открыл вклад')
plt.ylabel('Количество клиентов')
plt.show()

---

**Вычислите процент клиентов, открывших вклад**

In [ ]:
yes_percent = bank['target'].mean() * 100
print(f'Процент клиентов, открывших вклад: {yes_percent:.2f}%')

**Вопрос:** Сбалансированы ли классы? Какой класс преобладает?

**Ответ:** Классы несбалансированы. Преобладает класс `no`: около 88,5% клиентов не открыли вклад, и только около 11,5% открыли.

---

### ****3.2. Анализ пропущенных значений****

> 📚 **Подсказка:** Проверка пропусков с помощью `isna().sum()` показана в [**Занятии 1, Часть 3.6**](https://colab.research.google.com/drive/1Cnp7hPUUu5dorlhPPHUeztnnoq5Z2j4W?usp=sharing#scrollTo=1f244634) и [**Занятии 2, Часть 2.2**](https://colab.research.google.com/drive/1aT36P0Jc-dIjYOx6Hh8yecFSgt-z3Fan?usp=sharing#scrollTo=e96e5f15).

**Проверьте наличие пропущенных значений в датасете с помощью `isna().sum()`**

In [ ]:
bank.isna().sum()

**Вычислите процент пропущенных значений в каждом столбце датасета**

In [ ]:
missing_percent = bank.isna().mean() * 100
missing_percent.sort_values(ascending=False)

**Постройте столбчатую диаграмму (`barplot`) для визуализации процента пропусков**

In [ ]:
missing_percent_plot = missing_percent[missing_percent > 0].sort_values(ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(x=missing_percent_plot.index, y=missing_percent_plot.values)
plt.title('Процент пропущенных значений по столбцам')
plt.xlabel('Столбец')
plt.ylabel('Пропуски, %')
plt.xticks(rotation=45)
plt.show()

**Вопрос:** В каких столбцах больше всего пропусков? Какой процент данных потеряем, если удалим все строки с пропусками?

**Ответ:** Больше всего пропусков в `poutcome`, затем в `contact`, `education` и `job`. Если удалить все строки с пропусками, потеряем примерно 83% данных.

---

### ****3.3. Анализ числовых признаков****

> 📚 **Подсказка:** Построение гистограмм с разбивкой по целевой переменной (`hue`) показано в [**Занятии 2, Часть 2.4**](https://colab.research.google.com/drive/1aT36P0Jc-dIjYOx6Hh8yecFSgt-z3Fan?usp=sharing#scrollTo=ea28aa58).

**Обязательно постройте гистограммы для следующих признаков:**
- `age` — возраст
- `balance` — баланс на счёте  
- `duration` — длительность звонка

**Постройте гистограммы для числовых признаков с разбивкой по целевой переменной**

> Используйте `sns.histplot()` с параметром `hue='target'`

In [ ]:
numeric_features = ['age', 'balance', 'duration']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, col in enumerate(numeric_features):
    sns.histplot(data=bank, x=col, hue='target', kde=True, ax=axes[i])
    axes[i].set_title(f'Распределение {col}')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Количество')

plt.tight_layout()
plt.show()

*(При желании, можете не использовать мои шаблоны, а написать код для построения графиков с нуля и самостоятельно)*

**Вопрос:** Какие наблюдения вы можете сделать? Какие признаки различаются для клиентов, открывших и не открывших вклад?

**Ответ:** Сильнее всего различается `duration`: у клиентов, открывших вклад, звонки обычно дольше. `balance` различается слабее, но у клиентов с положительным и более высоким балансом вероятность открытия вклада выше. По `age` различия умеренные: чаще откликаются отдельные возрастные группы, но признак не так явно разделяет классы, как `duration`.

---

### ****Задание 3.4. Анализ категориальных признаков****

> 📚 **Подсказка:** Построение столбчатой диаграммы (`barplot`) для анализа связи категориальных признаков с целевой переменной показано в [**Занятии 2, Часть 2.5**](https://colab.research.google.com/drive/1aT36P0Jc-dIjYOx6Hh8yecFSgt-z3Fan?usp=sharing#scrollTo=ea6a781f).

**Постройте графики для анализа доли открывших депозит по категориальным признакам ('job', 'marital', 'education', 'contact'):**

In [ ]:
cat_features = ['job', 'marital', 'education', 'contact']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(cat_features):
    sns.barplot(data=bank, x=col, y='target', ax=axes[i])
    axes[i].set_title(f'Доля открывших вклад по {col}')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Доля target=1')
    axes[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

---

**Ответьте на вопросы:**

**Вопрос:** Люди каких профессий чаще открывают депозит?

**Ответ:** Чаще открывают депозит клиенты из групп `student` и `retired`; также относительно неплохая доля у `management`.

**Вопрос:** Влияет ли семейное положение на вероятность открытия депозита?

**Ответ:** Да, влияет: у `single` доля открывших вклад выше, чем у `married`; у `divorced` результат промежуточный.

**Вопрос:** Какой тип связи (contact) наиболее эффективен?

**Ответ:** Наиболее эффективен `cellular`.

**Вопрос:** Какие в общем категории клиентов чаще открывают вклад (примерный портрет по всем категориальным признакам в совокупности)?

**Ответ:** Чаще открывают вклад клиенты, с которыми связывались по мобильному телефону, чаще одинокие, а по профессии — студенты или пенсионеры; уровень образования чаще не ниже среднего/высшего.

---

### ****Задание 3.5. Анализ связей между признаками (Boxplot)****

> 📚 **Подсказка:** Использование boxplot для анализа распределений показано в [**Занятии 2, Часть 2.6**](https://colab.research.google.com/drive/1aT36P0Jc-dIjYOx6Hh8yecFSgt-z3Fan?usp=sharing#scrollTo=592b07cc).

**Постройте `boxplot` для анализа баланса по уровню образования**

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=bank, x='education', y='balance', showfliers=False)
plt.title('Баланс по уровню образования')
plt.xlabel('Образование')
plt.ylabel('Баланс')
plt.show()

**Вопрос:** Различается ли баланс у клиентов с разным образованием?

**Ответ:** Да, различия есть, но они не очень сильные. У клиентов с `tertiary` образованием медианный баланс обычно выше, однако внутри каждой группы большой разброс.

---

### ****Задание 3.6. Группировка данных****

> 📚 **Подсказка:** Метод `groupby()` подробно разобран в [**Занятии 1, Часть 3.7**](https://colab.research.google.com/drive/1Cnp7hPUUu5dorlhPPHUeztnnoq5Z2j4W?usp=sharing#scrollTo=baa512e4).

**Вычислите средний баланс и долю открывших депозит по типу работы**

In [ ]:
job_group = bank.groupby('job').agg(
    mean_balance=('balance', 'mean'),
    deposit_rate=('target', 'mean'),
    clients=('target', 'count')
).sort_values('deposit_rate', ascending=False)

job_group

**Вычислите долю открывших депозит по уровню образования**

In [ ]:
education_rate = bank.groupby('education')['target'].mean().sort_values(ascending=False)
education_rate

---

## ****Часть 4. Формулировка гипотез и план экспериментов****

> 📚 **Подсказка:** Формулировка гипотез на основе EDA описана в [**Занятии 2, Часть 3**](https://colab.research.google.com/drive/1aT36P0Jc-dIjYOx6Hh8yecFSgt-z3Fan?usp=sharing#scrollTo=329f49e7).

На основе проведённого EDA сформулируйте **минимум 3 гипотезы** о том, какие признаки и преобразования могут улучшить модель.

### ****Ваши гипотезы:****

(Дважды нажмите правой кнопкой мыши на ячейку ниже, чтобы вписать свои гипотезы)

| № | Наблюдение из EDA | Гипотеза | Как проверить | Метрика для сравнения |
|---|-------------------|----------|---------------|----------------------|
| 1 | `duration` заметно различается у классов | Длительность звонка будет одним из самых важных признаков | Сравнить модели с этим признаком и без него | F1, ROC-AUC |
| 2 | `pdays = -1` означает, что клиента ранее не контактировали | Признак `was_contacted` улучшит модель | Добавить бинарный признак `was_contacted` | Accuracy, F1 |
| 3 | В `contact` и `poutcome` много пропусков | Сам факт пропуска может быть информативен | Создать признаки `*_unknown` и сравнить качество | Accuracy, F1 |

**Примеры формулировок (для ориентира):**

| Наблюдение | Гипотеза | Как проверить | Метрика |
|------------|----------|---------------|---------|
| `duration` сильно различается у классов | Признак важен для предсказания | Сравнить accuracy с `duration` и без него | Accuracy, F1 |
| `pdays` = -1 означает "не контактировали" | Создать бинарный признак `was_contacted` | Добавить признак, сравнить accuracy | Accuracy, F1 |
| `balance` имеет отрицательные значения | Создать признак `has_positive_balance` | Добавить признак, сравнить accuracy | Accuracy, F1 |
| В `job` есть пропуски | Факт пропуска может быть информативен | Создать `job_unknown`, сравнить accuracy | Accuracy, F1 |
| Много категориальных признаков | One-Hot Encoding улучшит модель | `pd.get_dummies()`, сравнить accuracy | Accuracy, F1 |

> 📚 **Подсказка:** Методология проверки гипотез через эксперименты подробно разобрана в [**Занятии 2, Часть 6**](https://colab.research.google.com/drive/1aT36P0Jc-dIjYOx6Hh8yecFSgt-z3Fan?usp=sharing#scrollTo=28a74583).

---

## ****Часть 5. Сравнение методов классификации****

> 📚 **Подсказка:** Различные методы классификации (логистическая регрессия, деревья решений, ансамбли, SVM, KNN) подробно разбирались на занятиях по supervised learning.

### ****5.1. Расширенная функция оценки модели****

> 📚 **Подсказка:** Базовая функция `evaluate_model()` была создана в [**Занятии 2, после Части 4.6**](https://colab.research.google.com/drive/1aT36P0Jc-dIjYOx6Hh8yecFSgt-z3Fan?usp=sharing#scrollTo=maxKagcsxhv8).

In [ ]:
def evaluate_model_advanced(model, X, y, model_name="Model"):
    """
    Расширенная оценка модели с множественными метриками.

    Параметры:
    ----------
    model : sklearn estimator
        Модель для обучения и оценки
    X : DataFrame или array
        Признаки
    y : Series или array
        Целевая переменная
    model_name : str
        Название модели для отображения

    Возвращает:
    -----------
    dict : словарь с метриками
    """
    # 1. Разделение данных (80% train, 20% test)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    # 2. Масштабирование (ВАЖНО: fit только на train!)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # 3. Обучение модели
    model.fit(X_train_scaled, y_train)

    # 4. Предсказание
    y_pred = model.predict(X_test_scaled)

    # 5. Вероятности (если модель поддерживает)
    if hasattr(model, 'predict_proba'):
        y_proba = model.predict_proba(X_test_scaled)[:, 1]
        roc_auc = roc_auc_score(y_test, y_proba)
    else:
        y_proba = None
        roc_auc = None

    # 6. Расчёт метрик
    results = {
        'model_name': model_name,
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred),
        'roc_auc': roc_auc
    }

    return results, y_test, y_pred, y_proba

### ****5.2. Простая функция оценки (для быстрых экспериментов)****

In [ ]:
def evaluate_model(X, y):
    """
    Быстрая оценка модели логистической регрессии.
    Возвращает только accuracy.
    """
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    model = LogisticRegression(random_state=42, max_iter=1000)
    model.fit(X_train_scaled, y_train)

    y_pred = model.predict(X_test_scaled)
    accuracy = accuracy_score(y_test, y_pred)

    return accuracy

### ****5.3. Подготовка данных для baseline****

In [ ]:
# Выбираем только числовые признаки для baseline
numeric_cols = ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']

# Подготовьте X и y
X_baseline = bank[numeric_cols]
y_baseline = bank['target']

print(f"Размер X: {X_baseline.shape}")
print(f"Размер y: {y_baseline.shape}")

### ****5.4. Задание: Сравните ВСЕ методы классификации****

**Создайте словарь со всеми моделями:**

In [ ]:
# Словарь моделей для сравнения
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'AdaBoost': AdaBoostClassifier(random_state=42, algorithm='SAMME'),
    'SVM': SVC(probability=True, random_state=42),
    'KNN': KNeighborsClassifier(),
    'Naive Bayes': GaussianNB()
}

**Обучите все модели и соберите результаты:**

In [ ]:
# Словарь для хранения всех результатов экспериментов
all_results = {}

# Список для сбора результатов сравнения моделей
comparison_results = []

print("=" * 70)
print("СРАВНЕНИЕ ВСЕХ МЕТОДОВ КЛАССИФИКАЦИИ (BASELINE)")
print("=" * 70)

for name, model in models.items():
    results, _, _, _ = evaluate_model_advanced(model, X_baseline, y_baseline, name)
    comparison_results.append(results)
    all_results[name] = results
    roc_text = f"{results['roc_auc']:.4f}" if results['roc_auc'] is not None else 'N/A'
    print(f"{name:25} | Accuracy: {results['accuracy']:.4f} | F1: {results['f1']:.4f} | ROC-AUC: {roc_text}")

print("=" * 70)

<details>
<summary><b>Подсказка (код):</b></summary>

In [ ]:
comparison_results = []

print("=" * 70)
print("СРАВНЕНИЕ ВСЕХ МЕТОДОВ КЛАССИФИКАЦИИ (BASELINE)")
print("=" * 70)

for name, model in models.items():
    results, _, _, _ = evaluate_model_advanced(model, X_baseline, y_baseline, name)
    comparison_results.append(results)
    roc_text = f"{results['roc_auc']:.4f}" if results['roc_auc'] is not None else 'N/A'
    print(f"{name:25} | Accuracy: {results['accuracy']:.4f} | F1: {results['f1']:.4f} | ROC-AUC: {roc_text}")

print("=" * 70)

</details>

### ****5.5. Создайте DataFrame с результатами и визуализируйте****

In [ ]:
# Создайте DataFrame из comparison_results
df_comparison = pd.DataFrame(comparison_results)

# Отсортируйте по F1-score
df_comparison = df_comparison.sort_values('f1', ascending=False)

# Выведите таблицу
df_comparison

**Постройте визуализацию сравнения моделей:**

In [ ]:
# Постройте barplot для сравнения моделей по разным метрикам
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

metrics_to_plot = ['accuracy', 'f1', 'roc_auc']
titles = ['Accuracy', 'F1-Score', 'ROC-AUC']

for i, (metric, title) in enumerate(zip(metrics_to_plot, titles)):
    sns.barplot(data=df_comparison, x='model_name', y=metric, ax=axes[i])
    axes[i].set_title(title)
    axes[i].set_xlabel('Модель')
    axes[i].set_ylabel(metric)
    axes[i].tick_params(axis='x', rotation=90)

plt.tight_layout()
plt.show()

### ****5.6. Ответьте на вопросы****

**Вопрос:** Какие 3 модели показали лучший результат по F1-Score?

**Ответ:** Топ-3 модели — первые три строки таблицы `df_comparison`, отсортированной по `f1`.

**Вопрос:** Почему мы ориентируемся на F1-Score, а не на Accuracy? (Подсказка: вспомните баланс классов)

**Ответ:** Потому что классы сильно несбалансированы: большинство клиентов не открывают вклад. Accuracy может быть высокой даже у модели, которая почти всегда предсказывает `no`, а F1 учитывает и precision, и recall положительного класса.

**Вопрос:** Какая модель показала лучший ROC-AUC? Что это означает?

**Ответ:** Лучшая по ROC-AUC — модель с максимальным значением в столбце `roc_auc` таблицы `df_comparison`. Высокий ROC-AUC означает, что модель хорошо отделяет клиентов, которые откроют вклад, от клиентов, которые не откроют.

---

## ****Часть 6. Стратегии обработки пропусков****

> 📚 **Подсказка:** Стратегии заполнения пропусков (удаление, заполнение модой/медианой, создание признаков) подробно разобраны в [**Занятии 2, Часть 5**](https://colab.research.google.com/drive/1aT36P0Jc-dIjYOx6Hh8yecFSgt-z3Fan?usp=sharing#scrollTo=89cec4d9).

### ****6.1. Стратегия A: Удаление строк с пропусками****

In [ ]:
# Посмотрим, сколько данных потеряем
df_dropped = bank.dropna()

print(f"Было строк: {len(bank)}")
print(f"Стало строк: {len(df_dropped)}")
print(f"Потеряно: {len(bank) - len(df_dropped)} ({(len(bank) - len(df_dropped)) / len(bank) * 100:.1f}%)")

**Вопрос:** Приемлемо ли терять столько данных?

**Ответ:** Нет, терять около 83% строк неприемлемо: выборка станет слишком маленькой, и модель может хуже обобщать закономерности.

---

### ****6.2. Стратегия B: Заполнение модой****

> 📚 **Подсказка:** Заполнение пропусков модой через `fillna()` рассматривалось в [**Занятии 2, Часть 5, Стратегия B**](https://colab.research.google.com/drive/1aT36P0Jc-dIjYOx6Hh8yecFSgt-z3Fan?usp=sharing#scrollTo=8e470458).

**Заполните категориальные пропуски модой (самым частым значением)**

In [ ]:
df_filled = bank.copy()

cat_cols_with_na = ['job', 'education', 'contact', 'poutcome']

for col in cat_cols_with_na:
    if df_filled[col].isna().sum() > 0:
        mode_value = df_filled[col].mode()[0]
        df_filled[col] = df_filled[col].fillna(mode_value)

print(f"Пропусков после заполнения: {df_filled.isna().sum().sum()}")

### ****6.3. Стратегия C: Создание признаков из пропусков****

> 📚 **Подсказка:** Создание бинарных признаков из пропусков (например, `has_cabin` в Titanic) рассматривалось в [**Занятии 2, Часть 6, Эксперимент 2, Тест D**](https://colab.research.google.com/drive/1aT36P0Jc-dIjYOx6Hh8yecFSgt-z3Fan?usp=sharing#scrollTo=df9d9180).
>
> **Идея:** Сам факт того, что значение неизвестно, может быть информативен! Например: если клиент не указал профессию — это может что-то говорить о нём.

**Создайте признаки "был ли пропуск" ДО заполнения**

In [ ]:
df_fe = bank.copy()

df_fe['job_unknown'] = bank['job'].isna().astype(int)
df_fe['education_unknown'] = bank['education'].isna().astype(int)
df_fe['contact_unknown'] = bank['contact'].isna().astype(int)
df_fe['poutcome_unknown'] = bank['poutcome'].isna().astype(int)

df_fe[['job_unknown', 'education_unknown', 'contact_unknown', 'poutcome_unknown']].head()

---

## ****Часть 7. Feature Engineering и отбор признаков****

> 📚 **Подсказка:** Создание новых признаков на основе анализа данных описано в [**Занятии 2, Часть 6**](https://colab.research.google.com/drive/1aT36P0Jc-dIjYOx6Hh8yecFSgt-z3Fan?usp=sharing#scrollTo=28a74583).

### ****7.1. Создание новых признаков****

**Создайте следующие признаки на основе гипотез из EDA:**

In [ ]:
df_fe = bank.copy()

# 1. Был ли клиент контактирован ранее (pdays != -1)
df_fe['was_contacted'] = (bank['pdays'] != -1).astype(int)

# 2. Положительный баланс
df_fe['has_positive_balance'] = (bank['balance'] > 0).astype(int)

# 3. Признаки из пропусков (ДО заполнения!)
df_fe['job_unknown'] = bank['job'].isna().astype(int)
df_fe['education_unknown'] = bank['education'].isna().astype(int)
df_fe['contact_unknown'] = bank['contact'].isna().astype(int)

# Теперь заполняем пропуски
for col in ['job', 'education', 'contact', 'poutcome']:
    if df_fe[col].isna().sum() > 0:
        mode_value = df_fe[col].mode()[0]
        df_fe[col] = df_fe[col].fillna(mode_value)

print("Созданные признаки:")
print(df_fe[['was_contacted', 'has_positive_balance', 'job_unknown', 'education_unknown', 'contact_unknown']].head(10))

**Добавьте свои признаки на основе ваших гипотез:**

In [ ]:
# Дополнительные признаки
df_fe['long_call'] = (df_fe['duration'] > df_fe['duration'].median()).astype(int)
df_fe['many_contacts'] = (df_fe['campaign'] > 3).astype(int)
df_fe['balance_to_age'] = df_fe['balance'] / df_fe['age']

custom_features = ['long_call', 'many_contacts', 'balance_to_age']

df_fe[custom_features].head()

### ****7.2. Систематический перебор комбинаций признаков****

> **Цель:** Определить, какие признаки улучшают модель, а какие вносят шум.

**Определите базовый набор признаков и признаки для тестирования:**

In [ ]:
# Базовые числовые признаки
base_features = ['age', 'balance', 'duration', 'campaign', 'pdays', 'previous', 'day']

# Новые созданные признаки для тестирования
new_features = ['was_contacted', 'has_positive_balance', 'job_unknown',
                'education_unknown', 'contact_unknown'] + custom_features

# Оценим базовую модель
X_base = df_fe[base_features]
y = df_fe['target']

accuracy_base = evaluate_model(X_base, y)
print(f"Базовая модель ({len(base_features)} признаков): Accuracy = {accuracy_base:.4f}")
print(f"Признаки: {base_features}")
print("=" * 60)

**Протестируйте добавление каждого нового признака по отдельности:**

In [ ]:
feature_test_results = []

for feature in new_features:
    current_features = base_features + [feature]
    accuracy = evaluate_model(df_fe[current_features], y)
    feature_test_results.append({
        'feature': feature,
        'accuracy': accuracy,
        'delta_accuracy': accuracy - accuracy_base
    })

feature_test_df = pd.DataFrame(feature_test_results).sort_values('delta_accuracy', ascending=False)
feature_test_df

**Вопрос:** Какие признаки улучшили модель? Какие ухудшили (внесли шум)?

**Ответ:** Улучшили модель признаки с положительным `delta_accuracy` в таблице `feature_test_df`. Признаки с отрицательным `delta_accuracy` ухудшили качество и могут считаться шумовыми. Обычно полезны `was_contacted` и `long_call`, а признаки пропусков могут давать слабый эффект или шум.


### ****7.3. Анализ важности признаков с помощью Random Forest****

In [ ]:
# Соберём все признаки
all_numeric_features = base_features + new_features
X_all = df_fe[all_numeric_features]

# Обучим Random Forest и посмотрим важность признаков
from sklearn.ensemble import RandomForestClassifier

# Масштабирование
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_all)

# Обучение
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_scaled, y)

# Важность признаков
importance_df = pd.DataFrame({
    'feature': all_numeric_features,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

print("ВАЖНОСТЬ ПРИЗНАКОВ (Random Forest)")
print("=" * 40)
print(importance_df.to_string(index=False))
print("=" * 40)

# Визуализация
plt.figure(figsize=(10, 6))
sns.barplot(data=importance_df, x='importance', y='feature')
plt.title('Feature Importance (Random Forest)')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

### ****7.4. Рекурсивное исключение признаков (RFE)****

> 📚 **Подсказка:** [RFE](https://habr.com/ru/companies/otus/articles/528676/) автоматически отбирает наиболее важные признаки, последовательно удаляя наименее значимые.

In [ ]:
from sklearn.feature_selection import RFE

# Применяем RFE для отбора лучших признаков
selector = RFE(
    estimator=LogisticRegression(max_iter=1000, random_state=42),
    n_features_to_select=7,  # Отберём 7 лучших признаков
    step=1
)

selector.fit(X_scaled, y)

# Какие признаки отобраны?
selected_features = [f for f, s in zip(all_numeric_features, selector.support_) if s]
rejected_features = [f for f, s in zip(all_numeric_features, selector.support_) if not s]

print("РЕЗУЛЬТАТЫ RFE")
print("=" * 40)
print(f"Отобранные признаки ({len(selected_features)}): {selected_features}")
print(f"Отвергнутые признаки ({len(rejected_features)}): {rejected_features}")
print("=" * 40)

# Сравним accuracy
X_selected = df_fe[selected_features]
accuracy_selected = evaluate_model(X_selected, y)
accuracy_all = evaluate_model(X_all, y)

print(f"\nAccuracy (все {len(all_numeric_features)} признаков): {accuracy_all:.4f}")
print(f"Accuracy (отобранные {len(selected_features)} признаков): {accuracy_selected:.4f}")
print(f"Разница: {accuracy_selected - accuracy_all:+.4f}")

**Вопрос:** Помог ли отбор признаков улучшить модель? Какие признаки оказались лишними?

**Ответ:** Это видно по строке `Разница` в результате RFE: если разница положительная, отбор помог; если отрицательная — ухудшил качество. Лишними считаются признаки из списка `rejected_features`.

---

## ****Часть 8. Кодирование категориальных признаков****

> 📚 **Подсказка:** One-Hot Encoding для категориальных признаков рассматривался в [**Занятии 2, Часть 6**](https://colab.research.google.com/drive/1aT36P0Jc-dIjYOx6Hh8yecFSgt-z3Fan?usp=sharing#scrollTo=28a74583).

### ****8.1. Примените One-Hot Encoding****

In [ ]:
# Определяем категориальные столбцы для кодирования
cat_cols_to_encode = ['job', 'marital', 'education', 'default', 'housing',
                       'loan', 'contact', 'month', 'poutcome']

# Применяем One-Hot Encoding
df_encoded = pd.get_dummies(df_fe, columns=cat_cols_to_encode, drop_first=True)

print(f"Было признаков: {df_fe.shape[1]}")
print(f"Стало признаков: {df_encoded.shape[1]}")

### ****8.2. Подготовьте финальный набор признаков****

In [ ]:
# Удаляем целевые переменные и служебные столбцы
cols_to_drop = ['y', 'target']
X_final = df_encoded.drop(columns=cols_to_drop)
y_final = df_encoded['target']

print(f"Финальный размер X: {X_final.shape}")
print(f"Количество признаков: {X_final.shape[1]}")

### ****8.3. Сравните модели на расширенном наборе признаков****

In [ ]:
print("=" * 70)
print("СРАВНЕНИЕ МОДЕЛЕЙ НА РАСШИРЕННОМ НАБОРЕ ПРИЗНАКОВ")
print("=" * 70)

comparison_results_extended = []

for name, model in models.items():
    # Создаём новый экземпляр модели
    if name == 'Logistic Regression':
        model = LogisticRegression(max_iter=1000, random_state=42)
    elif name == 'Decision Tree':
        model = DecisionTreeClassifier(random_state=42)
    elif name == 'Random Forest':
        model = RandomForestClassifier(n_estimators=100, random_state=42)
    elif name == 'Gradient Boosting':
        model = GradientBoostingClassifier(random_state=42)
    elif name == 'AdaBoost':
        model = AdaBoostClassifier(random_state=42, algorithm='SAMME')
    elif name == 'SVM':
        model = SVC(probability=True, random_state=42)
    elif name == 'KNN':
        model = KNeighborsClassifier()
    elif name == 'Naive Bayes':
        model = GaussianNB()

    results, _, _, _ = evaluate_model_advanced(model, X_final, y_final, name)
    comparison_results_extended.append(results)
    roc_text = f"{results['roc_auc']:.4f}" if results['roc_auc'] is not None else 'N/A'
    print(f"{name:25} | Accuracy: {results['accuracy']:.4f} | F1: {results['f1']:.4f} | ROC-AUC: {roc_text}")

print("=" * 70)

# Создайте DataFrame и отсортируйте по F1
df_comparison_extended = pd.DataFrame(comparison_results_extended)
df_comparison_extended = df_comparison_extended.sort_values('f1', ascending=False)
print("\nРАНЖИРОВАНИЕ ПО F1-SCORE:")
print(df_comparison_extended[['model_name', 'accuracy', 'f1', 'roc_auc']].to_string(index=False))

---

## ****Часть 9. Выбор топ-3 моделей****

### ****9.1. Определите топ-3 модели по F1-Score****

In [ ]:
# Выберите топ-3 модели из df_comparison_extended
top_3_models = df_comparison_extended.head(3)['model_name'].tolist()

print("ТОП-3 МОДЕЛИ ПО F1-SCORE:")
print("=" * 40)
for i, model_name in enumerate(top_3_models, 1):
    row = df_comparison_extended[df_comparison_extended['model_name'] == model_name].iloc[0]
    print(f"{i}. {model_name}: F1 = {row['f1']:.4f}, ROC-AUC = {row['roc_auc']:.4f}")
print("=" * 40)

### ****9.2. Постройте ROC-кривые для топ-3 моделей****

In [ ]:
from sklearn.metrics import roc_curve, auc

# Подготовка данных
X_train, X_test, y_train, y_test = train_test_split(
    X_final, y_final, test_size=0.2, random_state=42, stratify=y_final
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Словарь для топ-3 моделей
top_3_model_objects = {}
for name in top_3_models:
    if name == 'Logistic Regression':
        top_3_model_objects[name] = LogisticRegression(max_iter=1000, random_state=42)
    elif name == 'Random Forest':
        top_3_model_objects[name] = RandomForestClassifier(n_estimators=100, random_state=42)
    elif name == 'Gradient Boosting':
        top_3_model_objects[name] = GradientBoostingClassifier(random_state=42)
    elif name == 'AdaBoost':
        top_3_model_objects[name] = AdaBoostClassifier(random_state=42, algorithm='SAMME')
    elif name == 'Decision Tree':
        top_3_model_objects[name] = DecisionTreeClassifier(random_state=42)
    elif name == 'SVM':
        top_3_model_objects[name] = SVC(probability=True, random_state=42)
    elif name == 'KNN':
        top_3_model_objects[name] = KNeighborsClassifier()
    elif name == 'Naive Bayes':
        top_3_model_objects[name] = GaussianNB()

# Построение ROC-кривых
plt.figure(figsize=(10, 8))

for name, model in top_3_model_objects.items():
    model.fit(X_train_scaled, y_train)

    if hasattr(model, 'predict_proba'):
        y_proba = model.predict_proba(X_test_scaled)[:, 1]
    else:
        y_proba = model.decision_function(X_test_scaled)

    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc = auc(fpr, tpr)

    plt.plot(fpr, tpr, linewidth=2, label=f'{name} (AUC = {roc_auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random (AUC = 0.5)')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC-кривые для топ-3 моделей', fontsize=14)
plt.legend(loc='lower right', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### ****9.3. Постройте Confusion Matrix для топ-3 моделей****

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, (name, model) in enumerate(top_3_model_objects.items()):
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)

    cm = confusion_matrix(y_test, y_pred)

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i])
    axes[i].set_title(f'{name}')
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('Actual')

plt.tight_layout()
plt.show()

**Вопрос:** Какая модель лучше всего находит положительный класс (клиентов, которые откроют вклад)?

**Ответ:** Лучше всего положительный класс находит модель из топ-3 с наибольшим recall для класса `1` и наибольшим количеством True Positive в confusion matrix. По графикам ROC дополнительно нужно смотреть модель с наибольшей площадью AUC.

---

## ****Часть 10. Оптимизация гиперпараметров с GridSearchCV****

> 📚 **Подсказка:** GridSearchCV перебирает все комбинации гиперпараметров и выбирает лучшую с помощью кросс-валидации.

### ****10.1. Определите сетки гиперпараметров для топ-3 моделей****

In [ ]:
# Сетки гиперпараметров для разных моделей
param_grids = {
    'Logistic Regression': {
        'C': [0.1, 1, 10],
        'class_weight': [None, 'balanced']
    },
    'Decision Tree': {
        'max_depth': [3, 5, 10, None],
        'min_samples_split': [2, 10],
        'class_weight': [None, 'balanced']
    },
    'Random Forest': {
        'n_estimators': [100, 200],
        'max_depth': [None, 5, 10],
        'min_samples_leaf': [1, 3],
        'class_weight': [None, 'balanced']
    },
    'Gradient Boosting': {
        'n_estimators': [50, 100],
        'learning_rate': [0.05, 0.1],
        'max_depth': [2, 3]
    },
    'AdaBoost': {
        'n_estimators': [50, 100],
        'learning_rate': [0.05, 0.1]
    },
    'SVM': {
        'C': [0.1, 1, 10],
        'kernel': ['rbf'],
        'gamma': ['scale', 'auto'],
        'class_weight': [None, 'balanced']
    },
    'KNN': {
        'n_neighbors': [3, 5, 7, 11],
        'weights': ['uniform', 'distance']
    },
    'Naive Bayes': {
        'var_smoothing': [1e-9, 1e-8, 1e-7]
    }
}

def get_model_for_grid(name):
    if name == 'Logistic Regression':
        return LogisticRegression(max_iter=1000, random_state=42)
    if name == 'Decision Tree':
        return DecisionTreeClassifier(random_state=42)
    if name == 'Random Forest':
        return RandomForestClassifier(random_state=42)
    if name == 'Gradient Boosting':
        return GradientBoostingClassifier(random_state=42)
    if name == 'AdaBoost':
        return AdaBoostClassifier(random_state=42, algorithm='SAMME')
    if name == 'SVM':
        return SVC(probability=True, random_state=42)
    if name == 'KNN':
        return KNeighborsClassifier()
    if name == 'Naive Bayes':
        return GaussianNB()

### ****10.2. Запустите GridSearchCV для каждой модели из топ-3****

In [ ]:
# Результаты тюнинга
tuning_results = {}

print("=" * 70)
print("GRIDSEARCHCV ДЛЯ ТОП-3 МОДЕЛЕЙ")
print("=" * 70)

for name in top_3_models:
    model = get_model_for_grid(name)
    grid = GridSearchCV(
        estimator=model,
        param_grid=param_grids[name],
        scoring='f1',
        cv=3,
        n_jobs=-1
    )
    grid.fit(X_train_scaled, y_train)

    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test_scaled)

    tuning_results[name] = {
        'best_params': grid.best_params_,
        'best_cv_f1': grid.best_score_,
        'test_accuracy': accuracy_score(y_test, y_pred),
        'test_f1': f1_score(y_test, y_pred),
        'best_estimator': best_model
    }

    print(f"{name}")
    print(f"Лучшие параметры: {grid.best_params_}")
    print(f"CV F1: {grid.best_score_:.4f}")
    print(f"Test Accuracy: {tuning_results[name]['test_accuracy']:.4f}")
    print(f"Test F1: {tuning_results[name]['test_f1']:.4f}")
    print("-" * 70)

### ****10.3. Сравните результаты ДО и ПОСЛЕ тюнинга****

In [ ]:
print("=" * 70)
print("СРАВНЕНИЕ РЕЗУЛЬТАТОВ ДО И ПОСЛЕ ТЮНИНГА")
print("=" * 70)

comparison_tuning = []

for name in top_3_models:
    # Результаты до тюнинга
    before = df_comparison_extended[df_comparison_extended['model_name'] == name].iloc[0]

    # Результаты после тюнинга
    after = tuning_results.get(name, {})

    if after:
        comparison_tuning.append({
            'Model': name,
            'F1 (до)': before['f1'],
            'F1 (после)': after['test_f1'],
            'Δ F1': after['test_f1'] - before['f1'],
            'Accuracy (до)': before['accuracy'],
            'Accuracy (после)': after['test_accuracy'],
            'Δ Accuracy': after['test_accuracy'] - before['accuracy']
        })

df_tuning_comparison = pd.DataFrame(comparison_tuning)
print(df_tuning_comparison.to_string(index=False))
print("=" * 70)

**Вопрос:** Для какой модели тюнинг дал наибольшее улучшение?

**Ответ:** Наибольшее улучшение дала модель с максимальным значением `Δ F1` в таблице `df_tuning_comparison`.

**Вопрос:** Какие гиперпараметры оказали наибольшее влияние на качество?

**Ответ:** Наибольшее влияние оказали параметры, выбранные GridSearchCV в `best_params`: для ансамблей обычно это `n_estimators`, `max_depth`, `learning_rate` или `min_samples_leaf`; для SVM — `C`, `gamma` и `class_weight`; для логистической регрессии — `C` и `class_weight`.

---

## ****Часть 11. Финальное сравнение и выводы****

### ****11.1. Постройте итоговую таблицу всех экспериментов****

In [ ]:
# Соберите все эксперименты в одну таблицу
final_summary = []

# Baseline (только числовые признаки)
baseline_results = df_comparison.iloc[0]  # Лучшая модель на baseline
final_summary.append({
    'Эксперимент': 'Baseline (числовые признаки)',
    'Модель': baseline_results['model_name'],
    'Кол-во признаков': len(numeric_cols),
    'F1-Score': baseline_results['f1'],
    'Accuracy': baseline_results['accuracy']
})

# После Feature Engineering
extended_results = df_comparison_extended.iloc[0]
final_summary.append({
    'Эксперимент': 'Feature Engineering + One-Hot',
    'Модель': extended_results['model_name'],
    'Кол-во признаков': X_final.shape[1],
    'F1-Score': extended_results['f1'],
    'Accuracy': extended_results['accuracy']
})

# После тюнинга (лучшая модель)
best_tuned = max(tuning_results.items(), key=lambda x: x[1]['test_f1'])
final_summary.append({
    'Эксперимент': 'После GridSearchCV',
    'Модель': f"{best_tuned[0]} (tuned)",
    'Кол-во признаков': X_final.shape[1],
    'F1-Score': best_tuned[1]['test_f1'],
    'Accuracy': best_tuned[1]['test_accuracy']
})

df_final_summary = pd.DataFrame(final_summary)
print("=" * 80)
print("ИТОГОВАЯ ТАБЛИЦА ЭКСПЕРИМЕНТОВ")
print("=" * 80)
print(df_final_summary.to_string(index=False))
print("=" * 80)

### ****11.2. Визуализируйте прогресс улучшения модели****

In [ ]:
# График прогресса
plt.figure(figsize=(10, 6))

x = range(len(df_final_summary))
plt.plot(x, df_final_summary['F1-Score'], 'bo-', markersize=10, linewidth=2, label='F1-Score')
plt.plot(x, df_final_summary['Accuracy'], 'rs--', markersize=10, linewidth=2, label='Accuracy')

plt.xticks(x, df_final_summary['Эксперимент'], rotation=15, ha='right')
plt.xlabel('Этап эксперимента', fontsize=12)
plt.ylabel('Метрика', fontsize=12)
plt.title('Прогресс улучшения модели', fontsize=14)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Прирост
print(f"\nОБЩИЙ ПРИРОСТ:")
print(f"F1-Score: {df_final_summary.iloc[0]['F1-Score']:.4f} → {df_final_summary.iloc[-1]['F1-Score']:.4f} ({df_final_summary.iloc[-1]['F1-Score'] - df_final_summary.iloc[0]['F1-Score']:+.4f})")
print(f"Accuracy: {df_final_summary.iloc[0]['Accuracy']:.4f} → {df_final_summary.iloc[-1]['Accuracy']:.4f} ({df_final_summary.iloc[-1]['Accuracy'] - df_final_summary.iloc[0]['Accuracy']:+.4f})")

### ****11.3. Ответьте на итоговые вопросы****

---

**Вопрос 1:** Какая модель показала лучший финальный результат и почему, по вашему мнению?

**Ответ:** Лучший финальный результат показала модель из блока `best_tuned`, потому что после GridSearchCV она получила максимальный F1-score на тестовой выборке среди топ-3 моделей.

---

**Вопрос 2:** Какие признаки оказались шумовыми (ухудшали модель)?

**Ответ:** Шумовыми оказались признаки с отрицательным `delta_accuracy` в `feature_test_df`, а также признаки из списка `rejected_features` после RFE.

---

**Вопрос 3:** Насколько Feature Engineering улучшил результат по сравнению с baseline?

**Ответ:** Улучшение равно разнице между F1-Score строк `Feature Engineering + One-Hot` и `Baseline (числовые признаки)` в таблице `df_final_summary`.

---

**Вопрос 4:** Насколько GridSearchCV улучшил результат? Стоило ли это затраченного времени?

**Ответ:** Улучшение равно разнице между F1-Score после GridSearchCV и до тюнинга. Если прирост положительный, тюнинг был полезен; если прирост минимальный, для этой задачи достаточно базовых настроек модели.

---

**Вопрос 5:** Какие гиперпараметры оказали наибольшее влияние на качество лучшей модели?

**Ответ:** Наибольшее влияние оказали гиперпараметры из `tuning_results[best_model_name]['best_params']`, то есть те параметры, которые GridSearchCV выбрал для лучшей финальной модели.

---

**Вопрос 6:** Какие подтвердились из ваших гипотез, сформулированных в Части 4? Какие не подтвердились?

**Ответ:** Подтвердилась гипотеза о важности `duration`, а также гипотеза о полезности признака предыдущего контакта `was_contacted`, если он улучшил метрику. Гипотеза о признаках пропусков подтвердилась только для тех `*_unknown`, у которых положительный вклад; остальные признаки пропусков можно считать неподтверждёнными.

---

**Вопрос 7:** Что бы вы попробовали ещё для улучшения модели, если бы было больше времени?

**Ответ:** Я бы попробовал балансировку классов, подбор порога классификации, более широкий GridSearchCV/RandomizedSearchCV, CatBoost/LightGBM и более аккуратное кодирование категориальных признаков.

---

## ****Бонус: Сохранение лучшей модели****

In [ ]:
import joblib

# Сохраняем лучшую модель
best_model_name = best_tuned[0]
best_model = tuning_results[best_model_name]['best_estimator']

joblib.dump(best_model, 'best_bank_marketing_model.pkl')
joblib.dump(scaler, 'scaler.pkl')

print(f"Лучшая модель ({best_model_name}) сохранена в 'best_bank_marketing_model.pkl'")
print(f"Scaler сохранён в 'scaler.pkl'")